# Model Training Runner

This notebook drives `src/model_training.py` end-to-end.

**Steps covered:**
1. Resolve project root and configure paths
2. Verify data files exist
3. Run the training script
4. Inspect saved metrics

## Step 1 — Resolve project root and configure paths

`git rev-parse --show-toplevel` gives the absolute repo root regardless of where
Jupyter was launched, so all paths below are stable.

In [2]:
import subprocess
import sys
from pathlib import Path

from src.paths import MODEL_DATA_DIR, MODELS_DIR

PROJECT_ROOT = subprocess.check_output(
    ["git", "rev-parse", "--show-toplevel"], text=True
).strip()

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

TRAIN = MODEL_DATA_DIR / "df_train_final.parquet"
VALID = MODEL_DATA_DIR / "df_valid_final.parquet"
TEST = MODEL_DATA_DIR / "df_test_final.parquet"
OUT_DIR = str(MODELS_DIR)
RUN_OUTPUT_DIR = MODELS_DIR / "quick" / "latest"

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"OUT_DIR      : {OUT_DIR}")

PROJECT_ROOT : D:/AI/Real projects/Academic_Advisor
OUT_DIR      : D:\AI\Real projects\Academic_Advisor\models


## Step 2 — Verify data files exist

In [3]:
for path in [TRAIN, VALID, TEST]:
    p = Path(path)
    status = "OK" if p.exists() else "MISSING"
    print(f"[{status}] {path}")

[OK] D:\AI\Real projects\Academic_Advisor\data\model_data\df_train_final.parquet
[OK] D:\AI\Real projects\Academic_Advisor\data\model_data\df_valid_final.parquet
[OK] D:\AI\Real projects\Academic_Advisor\data\model_data\df_test_final.parquet


## Step 3 — Run the training script

Runs `src.model_training` in a child process. This lets the operating system reclaim all training RAM when the run exits, instead of leaving allocator caches in the Jupyter kernel.

> **Important behavior**
> - The script reads the existing `df_*_final.parquet` splits; it never rebuilds data.
> - Every training command trains both models from scratch. It does not load saved weights.
> - Training uses four LightGBM threads by default to keep the machine responsive.
> - Use an inference or analysis script when you only want to load a model and predict.
> - New quick-run artifacts are saved under `models/quick/latest`.

In [ ]:
import json

command = [
    sys.executable, "-m", "src.model_training",
    "--train", str(TRAIN),
    "--valid", str(VALID),
]
print("Running:", " ".join(command))
subprocess.run(command, cwd=PROJECT_ROOT, check=True)

In [ ]:
contract_path = RUN_OUTPUT_DIR / "feature_contract.json"
with open(contract_path) as f:
    run_contract = json.load(f)

print(f"TRAIN path actually read : {run_contract['train_path']}")
print(f"VALID path actually read : {run_contract['valid_path']}")
print(f"TEST policy              : {run_contract['test_policy']}")

assert run_contract["train_path"] == str(TRAIN), "run did not read the configured TRAIN path"
assert run_contract["valid_path"] == str(VALID), "run did not read the configured VALID path"

## Step 4 — Inspect saved metrics

In [5]:
metrics_path = RUN_OUTPUT_DIR / "metrics.json"
with open(metrics_path) as f:
    metrics = json.load(f)

print(json.dumps(metrics, indent=2))

{
  "m1_pass_classifier": {
    "train": {
      "auc": 0.9055,
      "avg_precision": 0.9764,
      "accuracy": 0.8891,
      "precision": 0.8862,
      "recall": 0.9961,
      "f1": 0.9379,
      "fail_precision": 0.9396,
      "fail_recall": 0.3216,
      "fail_f1": 0.4792,
      "brier": 0.0825,
      "confusion_matrix": {
        "tn": 22992,
        "fp": 48490,
        "fn": 1478,
        "tp": 377505
      }
    },
    "valid": {
      "auc": 0.7807,
      "avg_precision": 0.9677,
      "accuracy": 0.8896,
      "precision": 0.8987,
      "recall": 0.9883,
      "f1": 0.9414,
      "fail_precision": 0.2524,
      "fail_recall": 0.0342,
      "fail_f1": 0.0603,
      "brier": 0.0864,
      "confusion_matrix": {
        "tn": 553,
        "fp": 15594,
        "fn": 1638,
        "tp": 138312
      }
    },
    "test": {
      "auc": 0.7692,
      "avg_precision": 0.97,
      "accuracy": 0.8991,
      "precision": 0.9118,
      "recall": 0.9844,
      "f1": 0.9467,
      "fail_pre

## Alternative — run as CLI command

From the project root in a terminal (all arguments default to
`MODEL_DATA_DIR/df_{train,valid,test}_final.parquet` and `MODELS_DIR`):

```bash
python -m src.model_training
```

Pass `--train/--valid/--test/--out` only to override a default. Set
`ACADEMIC_ADVISOR_DATA_DIR` before running so `src/paths.py` resolves the
intended data root.